# LAB 08 — Tokenization, Subword Tokens, Special Tokens & Attention Mask

Notebook này thực hành các nội dung ở **Mục 5, 6, 7, 10 và 12.1** của đề bài:
1. Tải tokenizer của pretrained Transformer (`google/bert_uncased_L-2_H-128_A-2` — chuẩn WordPiece giống `bert-base-uncased`).
2. Chạy bài tập **Mục 10** với câu `"The cats are playing."`: ghi token sequence, token IDs, special tokens, attention mask và so sánh số lượng `words` / `tokens` / `subword tokens`.
3. Minh họa tách subword (`##...`), cặp câu với `[CLS]` / `[SEP]`, và token `[MASK]` / `[UNK]`.
4. Chạy **Mục 12.1** với batch tokenization (`padding=True`, `truncation=True`, `return_tensors="pt"`), kiểm tra `tokens.keys()` và giải thích `input_ids`, `attention_mask`.

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer

MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Model / Tokenizer:", MODEL_NAME)
print("Vocab size:", tokenizer.vocab_size)
print("Special tokens map:", tokenizer.special_tokens_map)
print(
    "Special token IDs:",
    {
        tok: tokenizer.convert_tokens_to_ids(tok)
        for tok in ["[CLS]", "[SEP]", "[PAD]", "[MASK]", "[UNK]"]
    },
)

/Users/user/.cache/codex-runtimes/codex-primary-runtime/dependencies/python/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model / Tokenizer: google/bert_uncased_L-2_H-128_A-2
Vocab size: 30522
Special tokens map: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
Special token IDs: {'[CLS]': 101, '[SEP]': 102, '[PAD]': 0, '[MASK]': 103, '[UNK]': 100}


## 1. Calculation Exercise 2 — Tokenize `"The cats are playing."` (Mục 10)

Theo Mục 10, ta không tự đoán trước khi chạy mà chạy trực tiếp qua tokenizer để ghi lại chính xác:
- Token sequence
- Token IDs
- Special tokens
- Attention mask
- So sánh số lượng **words**, **tokens**, và **subword tokens**.

In [2]:
sentence = "The cats are playing."
encoded = tokenizer(sentence, return_tensors="pt")

input_ids = encoded["input_ids"][0].tolist()
attention_mask = encoded["attention_mask"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)

special_set = set(tokenizer.all_special_tokens)
special_positions = [(i, tok, tid) for i, (tok, tid) in enumerate(zip(tokens, input_ids)) if tok in special_set]
non_special_tokens = [tok for tok in tokens if tok not in special_set]
# Subword continuation pieces in WordPiece start with "##"
subword_pieces = [tok for tok in non_special_tokens if tok.startswith("##")]
raw_words = sentence.split()

print("Input text:", repr(sentence))
print("Token sequence:", tokens)
print("Token IDs:", input_ids)
print("Special tokens (index, token, id):", special_positions)
print("Attention mask:", attention_mask)
print()
print(f"So sánh số lượng cho '{sentence}':")
print(f"  - Số words (tách theo khoảng trắng): {len(raw_words)} -> {raw_words}")
print(f"  - Tổng số tokens (bao gồm [CLS], [SEP]): {len(tokens)}")
print(f"  - Số content tokens (không tính special tokens): {len(non_special_tokens)} -> {non_special_tokens}")
print(f"  - Số subword tokens dạng '##' (continuation subwords): {len(subword_pieces)} -> {subword_pieces}")

df_single = pd.DataFrame({
    "position": list(range(len(tokens))),
    "token": tokens,
    "token_id": input_ids,
    "attention_mask": attention_mask,
    "is_special": [t in special_set for t in tokens],
})
df_single

Input text: 'The cats are playing.'
Token sequence: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
Token IDs: [101, 1996, 8870, 2024, 2652, 1012, 102]
Special tokens (index, token, id): [(0, '[CLS]', 101), (6, '[SEP]', 102)]
Attention mask: [1, 1, 1, 1, 1, 1, 1]

So sánh số lượng cho 'The cats are playing.':
  - Số words (tách theo khoảng trắng): 4 -> ['The', 'cats', 'are', 'playing.']
  - Tổng số tokens (bao gồm [CLS], [SEP]): 7
  - Số content tokens (không tính special tokens): 5 -> ['the', 'cats', 'are', 'playing', '.']
  - Số subword tokens dạng '##' (continuation subwords): 0 -> []


,position,token,token_id,attention_mask,is_special
0,0,[CLS],101,1,True
1,1,the,1996,1,False
2,2,cats,8870,1,False
3,3,are,2024,1,False
4,4,playing,2652,1,False
5,5,.,1012,1,False
6,6,[SEP],102,1,True


### Nhận xét cho câu `"The cats are playing."`
- Tokenizer của BERT (`uncased`) tự chuyển toàn bộ chữ hoa thành chữ thường (`"The"` -> `"the"`) và tách dấu chấm `.` thành một token riêng.
- Hai token `[CLS]` (ID `101`) ở đầu và `[SEP]` (ID `102`) ở cuối được tự động chèn thêm, nên dù câu chỉ có **4 words** (`The`, `cats`, `are`, `playing.`), chuỗi đầu ra có tới **7 tokens** (5 content tokens + 2 special tokens).
- Với câu đơn giản này, các từ `"cats"` và `"playing"` đều có sẵn nguyên vẹn trong bộ từ điển 30,522 token của BERT nên có **0 subword continuation (`##`)**. Để thấy rõ cơ chế tách subword của WordPiece (Mục 5), ta thử với một câu chứa các từ ghép/ít gặp hơn ở cell ngay dưới đây.

In [3]:
demo_subword = "The unbelievable cats are playfully overacting."
sub_enc = tokenizer(demo_subword, return_tensors="pt")
sub_tokens = tokenizer.convert_ids_to_tokens(sub_enc["input_ids"][0].tolist())
sub_non_special = [t for t in sub_tokens if t not in special_set]
sub_continuations = [t for t in sub_non_special if t.startswith("##")]

print("Text:", repr(demo_subword))
print("Words:", len(demo_subword.split()), "->", demo_subword.split())
print("Tokens (total):", len(sub_tokens), "->", sub_tokens)
print("Content tokens:", len(sub_non_special))
print("Subword '##' tokens:", len(sub_continuations), "->", sub_continuations)
print("Ví dụ tách từ 'unbelievable':", tokenizer.tokenize("unbelievable"))
print("Ví dụ tách từ 'overacting':", tokenizer.tokenize("overacting"))

Text: 'The unbelievable cats are playfully overacting.'
Words: 6 -> ['The', 'unbelievable', 'cats', 'are', 'playfully', 'overacting.']
Tokens (total): 11 -> ['[CLS]', 'the', 'unbelievable', 'cats', 'are', 'playfully', 'over', '##act', '##ing', '.', '[SEP]']
Content tokens: 9
Subword '##' tokens: 2 -> ['##act', '##ing']
Ví dụ tách từ 'unbelievable': ['unbelievable']
Ví dụ tách từ 'overacting': ['over', '##act', '##ing']


## 2. Minh họa đầy đủ 5 Special Tokens (Mục 6)

Bảng các special tokens thường gặp trong BERT:
- **`[CLS]`** (ID `101`): đứng đầu sequence, embedding đầu ra tại vị trí này đại diện cho toàn bộ câu trong bài toán classification.
- **`[SEP]`** (ID `102`): đánh dấu kết thúc câu hoặc phân tách giữa câu A và câu B.
- **`[PAD]`** (ID `0`): đệm vào cuối các câu ngắn để mọi câu trong cùng batch có cùng độ dài tensor.
- **`[MASK]`** (ID `103`): che token trong giai đoạn pretraining bằng Masked Language Modeling (MLM).
- **`[UNK]`** (ID `100`): token thay thế khi gặp ký tự/từ hoàn toàn không nằm trong từ điển kể cả ở mức subword.

In [4]:
pair_enc = tokenizer("This movie is great.", "The acting is superb.", return_tensors="pt")
print("Pair tokens:", tokenizer.convert_ids_to_tokens(pair_enc["input_ids"][0].tolist()))
print("Pair token_type_ids:", pair_enc["token_type_ids"][0].tolist())

mlm_text = "The cat [MASK] fish."
print("MLM tokens:", tokenizer.convert_ids_to_tokens(tokenizer(mlm_text)["input_ids"]))

unk_text = "Strange symbol: ☃"
print("UNK demo:", tokenizer.convert_ids_to_tokens(tokenizer(unk_text)["input_ids"]))

Pair tokens: ['[CLS]', 'this', 'movie', 'is', 'great', '.', '[SEP]', 'the', 'acting', 'is', 'superb', '.', '[SEP]']
Pair token_type_ids: [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
MLM tokens: ['[CLS]', 'the', 'cat', '[MASK]', 'fish', '.', '[SEP]']
UNK demo: ['[CLS]', 'strange', 'symbol', ':', '[UNK]', '[SEP]']


## 3. Core Implementation — Batch Tokenization, Padding & Truncation (Mục 7 & 12.1)

Khi xử lý nhiều câu cùng lúc (batch), các câu có độ dài khác nhau nên cần `padding=True` và `truncation=True` với `return_tensors="pt"`.

In [5]:
texts = [
    "The cats are playing.",
    "I really loved this movie because the storyline and acting were surprisingly good!",
]

tokens = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

print("tokens.keys():", list(tokens.keys()))
for k, v in tokens.items():
    print(f"  {k}: shape={tuple(v.shape)}, dtype={v.dtype}")

for idx, txt in enumerate(texts):
    ids = tokens["input_ids"][idx].tolist()
    mask = tokens["attention_mask"][idx].tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"\n--- Sample {idx} ---")
    print("Text:", txt)
    print("Tokens:", toks)
    print("input_ids:", ids)
    print("attention_mask:", mask)

# Demo truncation khi max_length nhỏ hơn độ dài câu dài
tokens_trunc = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=10,
    return_tensors="pt",
)
print("\n--- Truncation demo (max_length=10) ---")
print("Shape:", tuple(tokens_trunc["input_ids"].shape))
print("Sample 1 truncated tokens:", tokenizer.convert_ids_to_tokens(tokens_trunc["input_ids"][1].tolist()))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


tokens.keys(): ['input_ids', 'token_type_ids', 'attention_mask']
  input_ids: shape=(2, 16), dtype=torch.int64
  token_type_ids: shape=(2, 16), dtype=torch.int64
  attention_mask: shape=(2, 16), dtype=torch.int64

--- Sample 0 ---
Text: The cats are playing.
Tokens: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
input_ids: [101, 1996, 8870, 2024, 2652, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]

--- Sample 1 ---
Text: I really loved this movie because the storyline and acting were surprisingly good!
Tokens: ['[CLS]', 'i', 'really', 'loved', 'this', 'movie', 'because', 'the', 'storyline', 'and', 'acting', 'were', 'surprisingly', 'good', '!', '[SEP]']
input_ids: [101, 1045, 2428, 3866, 2023, 3185, 2138, 1996, 9994, 1998, 3772, 2020, 10889, 2204, 999, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

--- Truncation demo

### Giải thích `tokens.keys()`, `input_ids` và `attention_mask` (Mục 7 & 12.1)

1. **`tokens.keys()`** trả về `['input_ids', 'token_type_ids', 'attention_mask']`:
   - **`input_ids`**: Ma trận số nguyên kích thước `(batch_size, seq_len)` chứa chỉ số (ID) của từng token trong bảng từ vựng (`30,522` tokens) để tra bảng Embedding của Transformer. Với Sample 0 ngắn hơn Sample 1, các vị trí dư ra ở cuối được điền ID `0` (`[PAD]`).
   - **`attention_mask`**: Ma trận nhị phân cùng kích thước `(batch_size, seq_len)`, trong đó `1` đánh dấu token thật (bao gồm `[CLS]`, nội dung, `[SEP]`) và `0` đánh dấu vị trí `[PAD]`. Trong Self-Attention, các vị trí có `attention_mask == 0` sẽ bị gán giá trị âm rất lớn (khoảng `-1e9`) trước hàm `softmax`, khiến attention weight đổ vào `[PAD]` bằng `0` — nhờ vậy **padding token không tham gia vào ngữ nghĩa thực của câu**.
   - **`token_type_ids`**: Đánh dấu câu thuộc phân đoạn `0` hay `1` (hữu ích khi nhập cặp câu như NLI hoặc QA; với single-text classification toàn bộ bằng `0`).